# Notebook 05 — The Timing: When During Pregnancy Exposure Matters
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> 📋 Health records → 🛰 Satellite → 🌍 Environment → 🔗 Link → [⏱ Timing] → 🎯 Results & GeoAI
> ```
>
> Every birth record now has climate values attached — but they are annual means.
> A heat anomaly in the first trimester is biologically different from the same anomaly
> in the third trimester. This notebook constructs **trimester-specific exposure windows**
> so that the timing of environmental stress is captured, not just its average level.

---

In [ ]:
%%html
<div id="nb05-prereq-gate" style="font-family:system-ui,sans-serif;margin:0 0 16px"></div>
<script>
(function(){
  var PREV_KEY='nb04_passed';
  var PREV_LABEL='Module 4: Spatial Join';
  var gate=document.getElementById('nb05-prereq-gate');
  if(!gate)return;
  if(!localStorage.getItem(PREV_KEY)){
    gate.innerHTML='<div style="background:#fff3cd;border:1px solid #ffc107;border-left:5px solid #ffc107;border-radius:6px;padding:14px 18px;color:#664d03"><b>&#9888; Prerequisite not detected</b><br>It looks like you have not yet passed the quiz for <b>'+PREV_LABEL+'</b>. We recommend completing that module first before continuing here. Your facilitator can help if needed.</div>';
  }else{
    gate.innerHTML='<div style="background:#d1e7dd;border-left:5px solid #198754;border-radius:6px;padding:10px 16px;color:#0a3622"><b>&#10003; '+PREV_LABEL+' completed</b> &#8212; welcome to the next module.</div>';
  }
})();
</script>

---
## Section 1 — Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))  # makes shared_utils importable
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")


In [ ]:
# Load the daily exposure data — one row per birth × pregnancy day
daily = pd.read_csv(DATA_DIR / 'zw_births_daily_exposure.csv', parse_dates=['date'])
daily['birth_id'] = daily['caseid'].str.strip() + '__' + daily['bidx'].astype(str)

# Load birth metadata (health outcomes, demographics)
births_meta = pd.read_csv(DATA_DIR / 'zw_births_raw.csv')
births_meta['birth_id'] = births_meta['caseid'].str.strip() + '__' + births_meta['bidx'].astype(str)

print(f'Daily exposure rows: {len(daily):,}')
print(f'Unique births: {daily["birth_id"].nunique():,}')
print(f'Variables: {[c for c in daily.columns if c not in ["birth_id","caseid","bidx","cluster_id","Latitude","Longitude","date"]]}')
print(f'\nDate range: {daily["date"].min().date()} → {daily["date"].max().date()}')
print(f'\nSample (first birth):')
sample_id = daily['birth_id'].iloc[0]
print(daily[daily['birth_id']==sample_id][['date','temp_c','pm25','precipitation','ndvi']].head(6).to_string(index=False))

### About this dataset

`zw_births_daily_exposure.csv` contains **one row per pregnancy day per birth**. For each of the 5,794 DHS births, the exposure date runs from the first day of the conception month to the first day of the birth month — giving a daily record of the climate and air quality environment experienced throughout the pregnancy.

| Variable | Source | Unit |
|---|---|---|
| `temp_c` | ERA5 reanalysis (CAMS NC file) | °C |
| `dust_aod550` | CAMS dust aerosol optical depth | dimensionless |
| `pm25` | CAMS PM2.5 surface concentration | µg/m³ |
| `precipitation` | CHIRPS daily rainfall | mm |
| `ndvi` | MODIS daily / 16-day composite | 0–1 |

The health outcomes (`low_birth_weight`, `birth_weight_g`, `province`, etc.) come from `zw_births_raw.csv` and are kept separate until we need them. **This is good epidemiological practice** — keep exposures and outcomes in separate files and only merge at analysis time.

In [ ]:
print("Low birth weight prevalence:")
lbw_prev = births_meta['low_birth_weight'].mean()
print(f"  {lbw_prev:.3f}  ({lbw_prev*100:.1f}%)")
print()
print(f"Births with recorded birth weight: {births_meta['birth_weight_g'].notna().sum():,} of {len(births_meta):,}")
print()
print("Gestational distribution (conception month):")
print(births_meta['conception_month'].value_counts().sort_index().to_string())

In [ ]:
# Compute trimester-specific means from the daily data
# Trimester windows (days since conception):
#   T1 = days 0–89    (conception through week 13)
#   T2 = days 90–179  (weeks 14–26)
#   T3 = days 180+    (week 27 to birth)

def compute_trimester_col(group):
    """Given all daily rows for one birth, compute T1/T2/T3 means."""
    # Sort by date so day-number is sequential
    g = group.sort_values('date').copy()
    g['day_num'] = (g['date'] - g['date'].iloc[0]).dt.days
    
    t1 = g[g['day_num'] < 90]
    t2 = g[(g['day_num'] >= 90) & (g['day_num'] < 180)]
    t3 = g[g['day_num'] >= 180]
    
    env_vars = ['temp_c', 'dust_aod550', 'pm25', 'precipitation', 'ndvi']
    rec = {'birth_id': group.name}
    for var in env_vars:
        short = {'temp_c':'temp','dust_aod550':'dust','pm25':'pm25','precipitation':'rain','ndvi':'ndvi'}[var]
        rec[f'{short}_t1'] = t1[var].mean() if len(t1) else float('nan')
        rec[f'{short}_t2'] = t2[var].mean() if len(t2) else float('nan')
        rec[f'{short}_t3'] = t3[var].mean() if len(t3) else float('nan')
    return pd.Series(rec)

print('Computing trimester means for 5,794 births (this takes ~1 minute)...')
trimester_df = daily.groupby('birth_id', group_keys=False).apply(compute_trimester_col)
trimester_df = trimester_df.reset_index(drop=True)

# Merge with birth metadata
births = births_meta.merge(trimester_df, on='birth_id', how='left')

# Derived flags
# Quartile-based thresholds: robust regardless of dataset temperature/rainfall range
_temp_t3_p75 = births['temp_t3'].quantile(0.75)    # top 25% T3 heat
_rain_t3_p25 = births['rain_t3'].quantile(0.25)    # bottom 25% T3 rainfall
births['hot_t3']         = births['temp_t3'] > _temp_t3_p75
births['drought_exposed'] = births['rain_t3'] < _rain_t3_p25

print(f'Trimester dataset: {len(births):,} births × {len(births.columns)} columns')
print(f'Trimester columns: {[c for c in births.columns if any(c.endswith(f"_t{t}") for t in [1,2,3])]}')
print(f'\nT3 temperature: mean={births.temp_t3.mean():.1f}°C, 75th pct={_temp_t3_p75:.1f}°C')
print(f'hot_t3 (top quartile T3 heat):             {births.hot_t3.sum():,} births ({births.hot_t3.mean()*100:.1f}%)')
print(f'T3 rainfall 25th pct={_rain_t3_p25:.3f} mm/day')
print(f'drought_exposed (bottom quartile T3 rain): {births.drought_exposed.sum():,} births ({births.drought_exposed.mean()*100:.1f}%)')

---
## Section 2 — Worked Example: Zimbabwe Seasonality and Pregnancy

For each conception month, we compute the mean third-trimester (T3) temperature across all births with that conception month. T3 = months 7–9 of gestation; a baby conceived in month `c` reaches T3 during calendar months `c+6` to `c+8` (wrapping around year-end).

Zimbabwe's rainy season runs approximately November–March. A baby conceived November–December experiences T3 in August–October — the transition from cool-dry to hot-dry — representing high cumulative heat stress.

**Why conception month?**  The month a pregnancy begins determines which climate
season each trimester falls in. A baby conceived in November will experience
T3 (weeks 27–40) during August–October — the hot-dry season before Zimbabwe's
rains begin. A baby conceived in May experiences T3 in February–April, the
tail of the rainy season when temperatures are lower and food availability higher.

The plot below makes this concrete: the x-axis is conception month, the y-axis
is the average T3 land surface temperature recorded for babies with that
conception month. Shaded bands mark Zimbabwe's rainy season (Nov–Mar).
Public-health researchers: this is analogous to timing-of-exposure analyses
used in nutritional epidemiology.

In [ ]:
RAINY_MONTHS = [11, 12, 1, 2, 3]  # Zimbabwe wet season

t3_by_conception = (
    births
    .groupby('conception_month')['temp_t3']
    .mean()
    .reset_index()
    .rename(columns={'temp_t3': 'mean_temp_t3'})
)

fig, ax = plt.subplots(figsize=(9, 4))

# Shade rainy season band
for m in RAINY_MONTHS:
    ax.axvspan(m - 0.5, m + 0.5, alpha=0.15, color='royalblue', zorder=0)

ax.plot(
    t3_by_conception['conception_month'],
    t3_by_conception['mean_temp_t3'],
    marker='o', ms=7, color='firebrick', lw=2.0, label='Mean T3 temperature'
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(
    ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)
ax.set_xlabel('Conception month', fontsize=11)
ax.set_ylabel('Mean T3 temperature (°C)', fontsize=11)
ax.set_title('Third-trimester heat by conception month — Zimbabwe DHS 2015', fontsize=12)

rainy_patch = mpatches.Patch(color='royalblue', alpha=0.3, label='Rainy season (Nov–Mar)')
ax.legend(handles=[ax.get_lines()[0], rainy_patch])
plt.tight_layout()
plt.show()

peak_c = int(t3_by_conception.loc[t3_by_conception['mean_temp_t3'].idxmax(), 'conception_month'])
print(f"Conception month with highest mean T3 temperature: month {peak_c}")
print("Interpretation: babies conceived in Nov–Dec reach T3 during Aug–Oct (hot-dry season).")

---
## Section 3 — Exercise 1 (3 pts): T3 Temperature by Birth Outcome

Among births with a recorded birth weight (`low_birth_weight` is not null), compute the mean third-trimester temperature for:
- LBW births (`low_birth_weight == 1`)
- Non-LBW births (`low_birth_weight == 0`)

**Store:**
- `temp_t3_lbw`             — float, 2 decimal places
- `temp_t3_normal`          — float, 2 decimal places
- `temp_diff_lbw_normal`   — float, 2 decimal places  (`temp_t3_lbw - temp_t3_normal`)

In [ ]:

births_bw = births.dropna(subset=['low_birth_weight']).copy()

temp_t3_lbw    = round(float(births_bw.loc[births_bw['low_birth_weight'] == 1, 'temp_t3'].mean()), 2)
temp_t3_normal = round(float(births_bw.loc[births_bw['low_birth_weight'] == 0, 'temp_t3'].mean()), 2)
temp_diff_lbw_normal = round(temp_t3_lbw - temp_t3_normal, 2)

print(f"Mean T3 temperature for LBW births:     {temp_t3_lbw:.2f} °C")
print(f"Mean T3 temperature for non-LBW births: {temp_t3_normal:.2f} °C")
print(f"Difference (LBW − normal):              {temp_diff_lbw_normal:+.2f} °C")

if temp_diff_lbw_normal > 0:
    print("\nLBW births experienced slightly higher T3 heat on average.")
else:
    print("\nLBW births experienced slightly lower T3 heat on average — other trimesters may be more important.")


In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Why might heat exposure in T3 (weeks 27 to birth) have a stronger effect on birth weight than identical heat exposure in T1 (weeks 1–13)?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> T3 is when the largest portion of fetal weight gain occurs and metabolic demands are highest</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Mothers spend more time outdoors in late pregnancy</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Heat only affects the fetus after 27 weeks gestation</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> T3 exposure data is more accurate than T1</span></label></div>
  <div id="nb05q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="A",E="The third trimester is the critical window for fetal growth. Heat impairs uteroplacental blood flow and increases maternal metabolic load, restricting nutrients available to the fetus precisely when weight gain is fastest.",ID="nb05q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Section 4 — Worked Example: Seasonality of Adverse Outcomes

We plot the monthly prevalence of LBW and neonatal death against birth month and overlay months with the highest mean T3 heat, to explore whether adverse outcomes cluster in particular seasons.

**Linking birth month to exposure:** if heat or drought in T3 drives LBW,
we expect LBW rates to peak among babies born ~3 months after the hottest
or driest period — because T3 precedes delivery by ~3 months. The dual-axis
chart plots birth month (not conception month) against LBW rate, so the
seasonal offset is baked in. Neonatal mortality is included as a second
outcome; it may track different exposure windows or have additional causes
(infection, healthcare access) that dilute the climate signal.

**GIS note:** birth month in DHS is recorded at the cluster level; spatial
clustering of births may amplify or suppress apparent seasonality.

In [ ]:
births_bw2 = births.dropna(subset=['low_birth_weight'])

monthly = (
    births_bw2
    .groupby('birth_month')
    .agg(
        n_births      = ('birth_weight_g', 'count'),
        lbw_rate      = ('low_birth_weight', 'mean'),
        neonatal_mr   = ('neonatal_death', 'mean'),
        mean_temp_t3  = ('temp_t3', 'mean')
    )
    .reset_index()
)

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax = axes[0]
ax.bar(monthly['birth_month'], monthly['lbw_rate'] * 100, color='steelblue',
       alpha=0.8, label='LBW rate (%)')
ax2 = ax.twinx()
ax2.plot(monthly['birth_month'], monthly['mean_temp_t3'], color='firebrick',
         marker='s', ms=5, lw=1.5, label='Mean T3 temperature (°C)')
ax.set_ylabel('LBW rate (%)', fontsize=10)
ax2.set_ylabel('Mean T3 temperature (°C)', fontsize=10, color='firebrick')
ax.set_title('Monthly LBW rate and third-trimester temperature', fontsize=11)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax = axes[1]
ax.bar(monthly['birth_month'], monthly['neonatal_mr'] * 1000, color='darkorange',
       alpha=0.8, label='Neonatal MR (per 1,000)')
ax.set_ylabel('Neonatal deaths per 1,000 births', fontsize=10)
ax.set_title('Monthly neonatal mortality rate', fontsize=11)
ax.legend(fontsize=9)

for a in axes:
    a.set_xticks(range(1, 13))
    a.set_xticklabels(month_labels)

axes[1].set_xlabel('Birth month', fontsize=11)
plt.tight_layout()
plt.show()

peak_lbw_month = int(monthly.loc[monthly['lbw_rate'].idxmax(), 'birth_month'])
print(f"Birth month with highest LBW rate: {month_labels[peak_lbw_month-1]} (month {peak_lbw_month})")

In [ ]:
import contextily as ctx
from pyproj import Transformer as _T

# Cluster-level LBW rate with coordinates (births has cluster_lat/lon from births_meta)
cluster_lbw = (
    births.dropna(subset=['low_birth_weight', 'cluster_lat', 'cluster_lon'])
    .groupby(['cluster_id', 'cluster_lat', 'cluster_lon'])
    .agg(
        lbw_rate  = ('low_birth_weight', 'mean'),
        mean_temp_t3 = ('temp_t3', 'mean'),
        n_births  = ('low_birth_weight', 'count'),
    )
    .reset_index()
)
cluster_lbw = cluster_lbw[cluster_lbw['n_births'] >= 3].copy()

_proj = _T.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
cx, cy = _proj.transform(cluster_lbw['cluster_lon'].values, cluster_lbw['cluster_lat'].values)
cluster_lbw['_mx'] = cx
cluster_lbw['_my'] = cy

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Panel 1 — LBW rate
ax = axes[0]
sc1 = ax.scatter(
    cluster_lbw['_mx'], cluster_lbw['_my'],
    c=cluster_lbw['lbw_rate'] * 100,
    cmap='YlOrRd', s=25, alpha=0.85, zorder=3
)
plt.colorbar(sc1, ax=ax, label='Cluster LBW rate (%)', shrink=0.65)
ax.set_title('Low Birth Weight Risk by Location\nZimbabwe DHS 2015', fontsize=11, fontweight='bold')
ax.set_axis_off()
try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception:
    pass

# Panel 2 — Mean T3 temperature
ax2 = axes[1]
sc2 = ax2.scatter(
    cluster_lbw['_mx'], cluster_lbw['_my'],
    c=cluster_lbw['mean_temp_t3'],
    cmap='RdYlBu_r', s=25, alpha=0.85, zorder=3
)
plt.colorbar(sc2, ax=ax2, label='Mean T3 temperature (°C)', shrink=0.65)
ax2.set_title('Third-Trimester Heat Exposure by Location\nZimbabwe DHS 2015', fontsize=11, fontweight='bold')
ax2.set_axis_off()
try:
    ctx.add_basemap(ax2, source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception:
    pass

plt.tight_layout()
plt.show()

print(f"Clusters shown: {len(cluster_lbw)}  (≥3 births per cluster)")
print(f"LBW rate range: {cluster_lbw['lbw_rate'].min()*100:.1f}% – {cluster_lbw['lbw_rate'].max()*100:.1f}%")
print(f"T3 temperature range: {cluster_lbw['mean_temp_t3'].min():.1f}°C – {cluster_lbw['mean_temp_t3'].max():.1f}°C")

---
## Section 5 — Exercise 2 (3 pts): Trimester Rainfall by Birth Outcome

Compute the mean T1, T2, and T3 rainfall separately for LBW births and non-LBW births. Store results as a DataFrame `rain_by_trimester_outcome` with:
- Index: `['t1', 't2', 't3']`
- Columns: `['lbw', 'normal']`

Then identify which trimester shows the largest absolute difference in rainfall between LBW and non-LBW births. Store the trimester name as `max_rain_diff_trimester` (string: `'t1'`, `'t2'`, or `'t3'`).

In [ ]:

births_bw3 = births.dropna(subset=['low_birth_weight']).copy()
lbw_births    = births_bw3[births_bw3['low_birth_weight'] == 1]
normal_births = births_bw3[births_bw3['low_birth_weight'] == 0]

rain_by_trimester_outcome = pd.DataFrame(
    {
        'lbw'   : [lbw_births['rain_t1'].mean(),    lbw_births['rain_t2'].mean(),    lbw_births['rain_t3'].mean()],
        'normal': [normal_births['rain_t1'].mean(), normal_births['rain_t2'].mean(), normal_births['rain_t3'].mean()],
    },
    index=['t1', 't2', 't3']
).round(2)

print("Mean rainfall (mm/mo) per trimester by birth outcome:")
print(rain_by_trimester_outcome)
print()

abs_diff = (rain_by_trimester_outcome['lbw'] - rain_by_trimester_outcome['normal']).abs()
max_rain_diff_trimester = abs_diff.idxmax()

print(f"Trimester with largest rainfall difference (LBW vs normal): {max_rain_diff_trimester.upper()}")
print(f"  LBW mean:    {rain_by_trimester_outcome.loc[max_rain_diff_trimester, 'lbw']:.2f} mm/mo")
print(f"  Normal mean: {rain_by_trimester_outcome.loc[max_rain_diff_trimester, 'normal']:.2f} mm/mo")


**What does trimester rainfall capture?**  Rainfall in this dataset comes from
CHIRPS (5 km resolution, ~monthly). For each birth, `rain_t1`, `rain_t2`, and
`rain_t3` represent the mean monthly rainfall over that trimester at the DHS cluster.
Rainfall acts as a proxy for multiple pathways: crop yields (food security),
standing water (infectious disease burden), and heat moderation. A single trimester
mean captures sustained wet or dry conditions but will miss a brief dry spell within
an otherwise rainy trimester — a limitation to keep in mind when interpreting results.

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 A baby is born in April. In which months did the mother's first trimester (T1, weeks 1–13) occur?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> January to March of the same year</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> July to September of the previous year</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> October to December of the previous year</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> April to June of the previous year</span></label></div>
  <div id="nb05q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Full-term delivery is ~40 weeks. April birth → conception ~July of the previous year. T1 (weeks 1–13) = approximately July–September. This falls in Zimbabwe\'s dry season — an important context for drought exposure.",ID="nb05q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Section 6 — Worked Example: Drought Exposure Analysis

The `drought_exposed` flag marks births in the **bottom quartile of third-trimester rainfall** — a relative indicator of gestational drought stress that is robust to inter-annual variability. We compare LBW rates, birth weights, and neonatal mortality between the lowest-rainfall and remaining births.

In [ ]:
# drought_exposed is stored as bool/string — normalise to bool
births['drought_exposed'] = births['drought_exposed'].astype(bool)

drought_yes = births[births['drought_exposed']]
drought_no  = births[~births['drought_exposed']]

summary = pd.DataFrame({
    'Group'              : ['Drought-exposed', 'Not drought-exposed'],
    'N births'           : [len(drought_yes), len(drought_no)],
    'LBW rate'           : [
        drought_yes['low_birth_weight'].mean(),
        drought_no['low_birth_weight'].mean()
    ],
    'Mean birth wt (g)'  : [
        drought_yes['birth_weight_g'].mean(),
        drought_no['birth_weight_g'].mean()
    ],
    'Neonatal MR'        : [
        drought_yes['neonatal_death'].mean(),
        drought_no['neonatal_death'].mean()
    ],
})

summary['LBW rate']         = summary['LBW rate'].round(3)
summary['Mean birth wt (g)'] = summary['Mean birth wt (g)'].round(1)
summary['Neonatal MR']      = summary['Neonatal MR'].round(3)

print("Drought exposure and birth outcomes — Zimbabwe DHS 2015:")
print(summary.to_string(index=False))
print()
print("Note: drought_exposed = T3 mean daily rainfall in the bottom quartile (≤ 25th percentile).")
print("This is an observational comparison; season, wealth, and geography confound the association.")

---
## Section 7 — Exercise 3 (2 pts): Third-Trimester NDVI and Birth Weight

Compute the Pearson correlation between T3 NDVI (`ndvi_t3`) and birth weight in grams (`birth_weight_g`) for births with a recorded birth weight. Store as `r_ndvi_t3_bw` (float, 3 decimal places).

NDVI is a proxy for vegetation cover and food availability — a greener environment during late pregnancy may signal better nutrition access and lower heat stress.

In [ ]:

births_complete = births.dropna(subset=['birth_weight_g', 'ndvi_t3'])

r_ndvi_t3_bw = round(
    float(births_complete['ndvi_t3'].corr(births_complete['birth_weight_g'])),
    3
)
print(f"Pearson r (T3 NDVI vs birth weight): {r_ndvi_t3_bw:.3f}")
print(f"N = {len(births_complete):,} births with both values recorded")

slope, intercept, *_ = stats.linregress(
    births_complete['ndvi_t3'], births_complete['birth_weight_g']
)
x_line = np.linspace(births_complete['ndvi_t3'].min(), births_complete['ndvi_t3'].max(), 100)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    births_complete['ndvi_t3'], births_complete['birth_weight_g'],
    s=6, alpha=0.25, color='forestgreen', label='Birth'
)
ax.plot(
    x_line, slope * x_line + intercept,
    color='firebrick', lw=2.0, label=f'OLS fit  (r = {r_ndvi_t3_bw})'
)
ax.set_xlabel('Third-trimester NDVI', fontsize=11)
ax.set_ylabel('Birth weight (g)', fontsize=11)
ax.set_title('T3 vegetation cover (NDVI) vs birth weight\nZimbabwe DHS 2015', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Sustained drought (SPEI < -1.5) during pregnancy may increase LBW risk. What is the most plausible biological pathway?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Drought increases ambient temperature which directly damages the fetus</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Drought reduces crop yields → food insecurity → maternal undernutrition → impaired fetal growth</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> SPEI directly measures caloric intake</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb05q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> Drought reduces hospital capacity</span></label></div>
  <div id="nb05q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="The pathway is mediated through food systems: drought → harvest failure → dietary diversity loss → maternal micronutrient deficiency → restricted intrauterine growth.",ID="nb05q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Recap

| Concept | Key point |
|---|---|
| Exposure timing | T1 = organogenesis; T3 = rapid fetal growth — different windows, different vulnerability |
| Trimester windows | DHS birth month − 9 = conception month; each trimester ≈ 3 calendar months |
| Seasonal patterning | Conception month determines which climate season each trimester falls in |
| Drought flag | Bottom quartile of T3 daily rainfall — relative indicator of gestational drought stress |
| T3 NDVI and birth weight | Positive correlation expected: greener = more food, lower heat stress |

**Next:** Notebook 06 is the culmination — province-level and cluster-level spatial maps, individual-level logistic regression with a forest plot, and a live GeoAI assistant that answers questions about the dataset in plain English.

In [ ]:
# Save trimester dataset to your personal home directory
# Each student's copy is independent — running this cell does not affect other students.
import pathlib as _pl
_home_out = _pl.Path.home() / 'zw_births_trimester.csv'
births.to_csv(_home_out, index=False)
print(f'Saved {len(births):,} birth records with trimester exposures')
print(f'Location: {_home_out}')
print('Notebook 06 will load this file automatically.')


In [ ]:
%%html
<div id="nb05-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 5 &#8212; Exposure Timeline Generation &#8212; Quiz Results</div>
  <div id="nb05-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb05-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:'nb05q1',correct:'A'},{id:'nb05q2',correct:'B'},{id:'nb05q3',correct:'B'}];
  var PASS=67;
  var NEXT='06_exposure_characterisation_geoai.ipynb';
  var NEXT_LABEL='Notebook 6: Results &amp; GeoAI';
  var NBID='nb05';
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem('mcq_'+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+'-score-display');
    var act=document.getElementById(NBID+'-next-action');
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions above to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML='';
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Your progress is saved &#8212; you can close and return to this notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+'_passed','true');
        if(NEXT){act.innerHTML='<a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px;margin-top:8px">&#10145; Continue to '+NEXT_LABEL+'</a>';}
        else{act.innerHTML='<div style="background:#cfe2ff;color:#084298;padding:12px 16px;border-radius:8px;margin-top:8px"><b>&#127881; You have completed the full GeoAI pipeline course! Congratulations.</b></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer, then your score updates automatically.</small></div>';
      if(act)act.innerHTML='';
    }
  }
  document.addEventListener('mcq-update',update);
  if(document.readyState==='loading')document.addEventListener('DOMContentLoaded',update);
  else update();
  setInterval(update,3000);
})();
</script>

---

## ✅ Notebook 05 complete

You now have trimester-specific exposures: T1, T2, T3 heat, drought, and rainfall for each birth record. The temporal dimension of environmental risk is captured.

➡️ **Notebook 06** is the culmination: spatial mapping, statistical modelling, and a live GeoAI assistant that lets you query the dataset in plain English. This is where the pipeline pays off.